# 3.0 — Théorie de l'information appliquée : entropie, KL, cross-entropy

**Navigation** : [<< 2.2 Descente de gradient](../02-ML-Cours/2.2-Descente-de-gradient.ipynb) · [2.3 Régression logistique](../02-ML-Cours/2.3-Regression-lineaire-logistique.ipynb) · [Feuille de route de la série](README.md) · [3.1 Rétropropagation >>](3.1-Retropropagation.ipynb)

**Kernel** : Python 3 · **Bibliothèques** : NumPy, matplotlib, scipy.special · **Niveau** : débutant en ML/IA (avant 3.1) · **CPU** : oui · **Temps d'exécution** : < 30 s


## Pourquoi ce notebook

L'**entropie croisée** et la **divergence KL** sont les fonctions de perte de tout le dépôt — GRPO (PT_04), RL distributional, classification QC — mais elles sont toujours invoquées comme des incantations, jamais construites depuis les données.

> Pourquoi la cross-entropy et pas la MSE pour la classification ? Pourquoi la KL dirige-t-elle un DPO ?

Ces questions méritent une réponse construite à la main, pas une formule recopiée. C'est ce que ce notebook fait : chaque définition est implémentée en NumPy, **vérifiée numériquement** (KL ≥ 0, H(p,q) = H(p) + KL(p||q), nulle ssi p=q), puis confrontée aux fonctions scipy qui l'ont précédée. La **discipline du dépôt** (from scratch puis framework) appliquée à la théorie de l'information.

**Plan en cinq actes** :

1. **Entropie from scratch** : compter des symboles dans du texte français réel, mesurer l'information avant de la définir formellement.
2. **Entropie croisée et KL** : deux distributions manipulées à la main, décomposition `H(p,q) = H(p) + KL(p||q)`, vérifications numériques.
3. **Pourquoi la cross-entropy pour la classification** : comparaison MSE vs cross-entropy sur la régression logistique de 2.3, la saturation du gradient `sigmoid+MSE` rendue visible.
4. **KL entre modèles** : compression vue comme modélisation, KL entre un modèle concentré et un modèle lissé.
5. **Ponts dépôt** : relire la loss DPO de PT_03, la KL de régularisation de PT_04/RL avec ces lunettes — extraits commentés.

**Prérequis** : [2.2 (descente de gradient)](../02-ML-Cours/2.2-Descente-de-gradient.ipynb) pour la dynamique d'optimisation, [2.3 (régression logistique)](../02-ML-Cours/2.3-Regression-lineaire-logistique.ipynb) pour le modèle qu'on réutilise. **À lire ensuite** : [3.1 (rétropropagation)](3.1-Retropropagation.ipynb) — la loss qu'on différentie.


## 1. Entropie from scratch : compter avant de définir

L'entropie d'une distribution discrète $p$ mesure l'**information moyenne** que porte un tirage aléatoire de $p$ :

$$H(p) = -\sum_{x} p(x) \log_2 p(x) \quad \text{(en bits)}$$

Trois propriétés qu'on va construire et vérifier :

- $H(p) \geq 0$ toujours.
- $H(p) = 0$ ssi $p$ est concentrée sur un seul symbole.
- $H(p) \leq \log_2 |\mathcal{X}|$ avec égalité ssi $p$ est uniforme — **l'uniformité maximise l'incertitude**.

Commençons par mesurer l'entropie d'un texte français réel, sans la définir formellement : on **compte** d'abord, on **interprète** après. Le corpus (Maupassant, *Le Horla*, 1887, domaine public) est embarqué dans la cellule suivante — le notebook est autonome.


In [1]:
# Cellule 1 — Corpus embarque : extrait du Horla (Maupassant, 1887, domaine public)
# Le corpus est petit (~5 ko) mais largement suffisant pour mesurer l'entropie
# d'un texte francais. Si tu veux un corpus plus gros, remplace par n'importe
# quel fichier utf-8 de ton choix (ex : un roman en .txt).

import collections
import math

CORPUS = "GUY DE MAUPASSANT\n\nLe Horla\n\n\n\n1887\n\n\n\n\nLE HORLA\n\n\n\n\n_8 mai._--Quelle journée admirable! J'ai passé toute la matinée étendu sur\nl'herbe, devant ma maison, sous l'énorme platane qui la couvre, l'abrite et\nl'ombrage tout entière. J'aime ce pays, et j'aime y vivre parce que j'y ai\nmes racines, ces profondes et délicates racines, qui attachent un homme à\nla terre où sont nés et morts ses aïeux, qui l'attachent à ce qu'on pense\net à ce qu'on mange, aux usages comme aux nourritures, aux locutions\nlocales, aux intonations des paysans, aux odeurs du sol, des villages et de\nl'air lui-même.\n\nJ'aime ma maison où j'ai grandi. De mes fenêtres, je vois la Seine qui\ncoule, le long de mon jardin, derrière la route, presque chez moi, la\ngrande et large Seine, qui va de Rouen au Havre, couverte de bateaux qui\npassent.\n\nA gauche, là-bas, Rouen, la vaste ville aux toits bleus, sous le peuple\npointu des clochers gothiques. Ils sont innombrables, frêles ou larges,\ndominés par la flèche de fonte de la cathédrale, et pleins de cloches qui\nsonnent dans l'air bleu des belles matinées, jetant jusqu'à moi leur doux\net lointain bourdonnement de fer, leur chant d'airain que la brise\nm'apporte, tantôt plus fort et tantôt plus affaibli, suivant qu'elle\ns'éveille ou s'assoupit.\n\nComme il faisait bon ce matin!\n\nVers onze heures, un long convoi de navires, traînés par un remorqueur,\ngros comme une mouche, et qui râlait de peine en vomissant une fumée\népaisse, défila devant ma grille.\n\nAprès deux goëlettes anglaises, dont le pavillon rouge ondoyait sur le\nciel, venait un superbe trois-mats brésilien, tout blanc, admirablement\npropre et luisant. Je le saluai, je ne sais pourquoi, tant ce navire me fit\nplaisir à voir.\n\n_12 mai_.--J'ai un peu de fièvre depuis quelques jours; je me sens\nsouffrant, ou plutôt je me sens triste.\n\nD'où viennent ces influences mystérieuses qui changent en découragement\nnotre bonheur et notre confiance en détresse. On dirait que l'air, l'air\ninvisible est plein d'inconnaissables Puissances, dont nous subissons les\nvoisinages mystérieux. Je m'éveille plein de gaîté, avec des envies de\nchanter dans la gorge.--Pourquoi?--Je descends le long de l'eau; et\nsoudain, après une courte promenade, je rentre désolé, comme si quelque\nmalheur m'attendait chez moi.--Pourquoi?--Est-ce un frisson de froid qui,\nfrôlant ma peau, a ébranlé mes nerfs et assombri mon âme? Est-ce la forme\ndes nuages, ou la couleur du jour, la couleur des choses, si variable, qui,\npassant par mes yeux, a troublé ma pensée? Sait-on? Tout ce qui nous\nentoure, tout ce que nous voyons sans le regarder, tout ce que nous frôlons\nsans le connaître, tout ce que nous touchons sans le palper, tout ce que\nnous rencontrons sans le distinguer, a sur nous, sur nos organes et, par\neux, sur nos idées, sur notre coeur lui-même, des effets rapides,\nsurprenants et inexplicables?\n\nComme il est profond, ce mystère de l'Invisible! Nous ne le pouvons sonder\navec nos sens misérables, avec nos yeux qui ne savent apercevoir ni le trop\npetit, ni le trop grand, ni le trop près, ni le trop loin, ni les habitants\nd'une étoile, ni les habitants d'une goutte d'eau... avec nos oreilles qui\nnous trompent, car elles nous transmettent les vibrations de l'air en notes\nsonores. Elles sont des fées qui font ce miracle de changer en bruit ce\nmouvement et par cette métamorphose donnent naissance à la musique, qui\nrend chantante l'agitation muette de la nature... avec notre odorat, plus\nfaible que celui du chien... avec notre goût, qui peut à peine discerner\nl'âge d'un vin!\n\nAh! si nous avions d'autres organes qui accompliraient en notre faveur\nd'autres miracles, que de choses nous pourrions découvrir encore autour de\nnous!\n\n_16 mai_.--Je suis malade, décidément! Je me portais si bien le mois\ndernier! J'ai la fièvre, une fièvre atroce, ou plutôt un énervement\nfiévreux, qui rend mon âme aussi souffrante que mon corps. J'ai sans cesse\ncette sensation affreuse d'un danger menaçant, cette appréhension d'un\nmalheur qui vient ou de la mort qui approche, ce pressentiment qui est sans\ndoute l'atteinte d'un mal encore inconnu, germant dans le sang et dans la\nchair.\n\n_18 mai_.--Je viens d'aller consulter mon médecin, car je ne pouvais plus\ndormir. Il m'a trouvé le pouls rapide, l'oeil dilaté, les nerfs vibrants,\nmais sans aucun symptôme alarmant. Je dois me soumettre aux douches et\nboire du bromure de potassium.\n\n_25 mai_.--Aucun changement! Mon état, vraiment, est bizarre. A mesure\nqu'approche le soir, une inquiétude incompréhensible m'envahit, comme si la\nnuit cachait pour moi une menace terrible. Je dîne vite, puis j'essaye de\nlire; mais je ne comprends pas les mots; je distingue à peine les lettres.\nJe marche alors dans mon salon de long en large, sous l'oppression d'une\ncrainte confuse et irrésistible, la crainte du sommeil et la crainte du\nlit.\n\nVers dix heures, je monte dans ma chambre. A peine entré, je donne deux\ntours de clef, et je pousse les verrous; j'ai peur... de quoi?... Je ne\nredoutais rien ju"

print(f"Longueur du corpus : {len(CORPUS):,} caracteres")
print(f"Apercu : {CORPUS[:120]!r}...")

# Comptage par caractere
counts = collections.Counter(CORPUS)
N = sum(counts.values())
print(f"\nNombre de caracteres distincts : {len(counts)}")
print(f"Total de tirages : {N:,}")

# Distribution empirique
probs = {ch: c / N for ch, c in counts.items()}
# Top 10
top = sorted(probs.items(), key=lambda x: -x[1])[:10]
print("\nTop 10 caracteres (frequence empirique) :")
for ch, p in top:
    label = repr(ch)[1:-1] if ch not in (' ', '\n', '\t') else {chr(32):'espace', chr(10):'\\n', chr(9):'\\t'}.get(ord(ch), repr(ch))
    print(f"  {label:8s} {p:.4f}")


Longueur du corpus : 5,000 caracteres
Apercu : "GUY DE MAUPASSANT\n\nLe Horla\n\n\n\n1887\n\n\n\n\nLE HORLA\n\n\n\n\n_8 mai._--Quelle journée admirable! J'ai passé toute la matinée éte"...

Nombre de caracteres distincts : 72
Total de tirages : 5,000

Top 10 caracteres (frequence empirique) :
  ' '      0.1504
  e        0.1106
  s        0.0632
  n        0.0626
  a        0.0576
  i        0.0516
  u        0.0502
  o        0.0498
  t        0.0496
  r        0.0486


In [2]:
# Cellule 2 — Entropie empirique du texte francais, et comparaison avec un texte aleatoire

# Entropie empirique du texte francais (sur le corpus du Horla)
H_fr = -sum(p * math.log2(p) for p in probs.values())
print(f"Entropie du texte francais : H = {H_fr:.4f} bits/caractere")
print(f"  -> Le francais porte {H_fr:.2f} bits d'information par caractere")
print(f"  -> Un caractere uniformement tire sur {len(counts)} symboles porterait log2({len(counts)}) = {math.log2(len(counts)):.4f} bits")

# Texte aleatoire : on genere N caracteres uniformes sur l'alphabet observe et on calcule son entropie
import random
random.seed(42)
alphabet = list(counts.keys())
random_text = "".join(random.choices(alphabet, k=N))
rand_counts = collections.Counter(random_text)
rand_probs = {ch: c / N for ch, c in rand_counts.items()}
H_rand = -sum(p * math.log2(p) for p in rand_probs.values())
print(f"\nEntropie du texte uniforme de meme longueur : H = {H_rand:.4f} bits/caractere")
print(f"  -> Proche de log2({len(rand_counts)}) = {math.log2(len(rand_counts)):.4f} (uniforme maximise l'entropie)")

# Texte redondant : 1000 fois le meme caractere (ici 'a')
mono = "a" * 1000
mono_counts = collections.Counter(mono)
mono_probs = {ch: c / 1000 for ch, c in mono_counts.items()}
H_mono = -sum(p * math.log2(p) for p in mono_probs.values())
print(f"\nEntropie d'un texte constant : H = {H_mono:.4f} bits/caractere (devrait etre 0)")

print(f"\nHierarchie : H_mono < H_francais < H_aleatoire (uniforme)")
print(f"             {H_mono:.3f}     {H_fr:.3f}     {H_rand:.3f}")


Entropie du texte francais : H = 4.5166 bits/caractere
  -> Le francais porte 4.52 bits d'information par caractere
  -> Un caractere uniformement tire sur 72 symboles porterait log2(72) = 6.1699 bits

Entropie du texte uniforme de meme longueur : H = 6.1612 bits/caractere
  -> Proche de log2(72) = 6.1699 (uniforme maximise l'entropie)

Entropie d'un texte constant : H = -0.0000 bits/caractere (devrait etre 0)

Hierarchie : H_mono < H_francais < H_aleatoire (uniforme)
             -0.000     4.517     6.161


## 2. Entropie croisée et divergence KL

Deux distributions $p$ (vraie) et $q$ (modèle). On mesure leur écart par la **divergence de Kullback-Leibler** :

$$D_{\mathrm{KL}}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)} = -H(p) + H(p, q)$$

où $H(p, q) = -\sum_x p(x) \log q(x)$ est l'**entropie croisée**. La décomposition **fondamentale** est :

$$\underbrace{H(p, q)}_{\text{cross-entropy}} = \underbrace{H(p)}_{\text{entropie de } p} + \underbrace{D_{\mathrm{KL}}(p \| q)}_{\text{écart à } p}$$

Trois propriétés qu'on vérifie numériquement :

- $D_{\mathrm{KL}}(p \| q) \geq 0$ avec **égalité ssi** $p = q$.
- La cross-entropy $H(p, q) \geq H(p)$ — le modèle est toujours au moins aussi "surpris" par la vérité qu'un oracle omniscient.
- Le minimum de $H(p, q)$ en $q$ est atteint en $q = p$ (c'est pour ça qu'on minimise la cross-entropy).


In [3]:
# Cellule 3 — KL et cross-entropy from scratch, avec epsilon-safe

import numpy as np

EPS = 1e-12  # evite log(0) ; le role de cette constante est explicite plus loin

def entropy(p, base=2):
    """H(p) = -sum p log_base p."""
    p = np.asarray(p, dtype=np.float64)
    assert np.all(p >= 0) and abs(p.sum() - 1.0) < 1e-9, "p doit etre une distribution de probabilites"
    return -float(np.sum(p * np.log(p) / np.log(base)))

def cross_entropy(p, q, base=2):
    """H(p, q) = -sum p log_base q (q est le modele)."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    assert np.all(p >= 0) and abs(p.sum() - 1.0) < 1e-9
    assert np.all(q >= 0) and abs(q.sum() - 1.0) < 1e-9
    # Epsilon-safe : on remplace 0 par EPS dans log pour eviter -inf
    q_safe = np.where(q == 0, EPS, q)
    return -float(np.sum(p * np.log(q_safe) / np.log(base)))

def kl_divergence(p, q, base=2):
    """D_KL(p || q) = sum p log_base (p / q)."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p_safe = np.where(p == 0, EPS, p)  # log(0/anything) -> 0 si p = 0
    q_safe = np.where(q == 0, EPS, q)
    return float(np.sum(p_safe * (np.log(p_safe) - np.log(q_safe)) / np.log(base)))

# Trois distributions-test
p = np.array([0.5, 0.3, 0.2])         # vraie
q1 = np.array([0.5, 0.3, 0.2])        # egale a p
q2 = np.array([0.4, 0.4, 0.2])        # proche
q3 = np.array([0.1, 0.1, 0.8])        # eloignee
q4 = np.array([0.99, 0.005, 0.005])   # tres concentree ailleurs

print("Verification de la decomposition H(p,q) = H(p) + KL(p||q)")
print("=" * 72)
H_p = entropy(p)
for label, q in [("q=p (identique)", q1), ("q proche de p", q2), ("q eloignee", q3), ("q tres eloignee", q4)]:
    H_pq = cross_entropy(p, q)
    KL_pq = kl_divergence(p, q)
    diff = H_pq - (H_p + KL_pq)
    print(f"{label:22s}  H(p,q)={H_pq:.6f}  H(p)={H_p:.6f}  KL={KL_pq:.6f}  ecart={diff:+.2e}")


Verification de la decomposition H(p,q) = H(p) + KL(p||q)
q=p (identique)         H(p,q)=1.485475  H(p)=1.485475  KL=0.000000  ecart=+0.00e+00
q proche de p           H(p,q)=1.521928  H(p)=1.485475  KL=0.036453  ecart=-2.22e-16
q eloignee              H(p,q)=2.721928  H(p)=1.485475  KL=1.236453  ecart=+0.00e+00
q tres eloignee         H(p,q)=3.829178  H(p)=1.485475  KL=2.343703  ecart=-4.44e-16


In [4]:
# Cellule 4 — Proprietes de la KL : non-negativite, minimum, non-symetrie

print("Propriete 1 : KL(p||q) >= 0, avec egalite ssi p = q")
print("-" * 60)
for label, (a, b) in [
    ("p = q",       (np.array([0.5, 0.3, 0.2]), np.array([0.5, 0.3, 0.2]))),
    ("p proche",    (np.array([0.5, 0.3, 0.2]), np.array([0.4, 0.4, 0.2]))),
    ("p different", (np.array([0.5, 0.3, 0.2]), np.array([0.1, 0.1, 0.8]))),
    ("p uniforme",  (np.array([1/3, 1/3, 1/3]), np.array([0.5, 0.3, 0.2]))),
]:
    print(f"  {label:18s}  KL = {kl_divergence(a, b):.6f}")

print("\nPropriete 2 : KL n'est PAS une distance (non symetrique)")
print("-" * 60)
p = np.array([0.9, 0.05, 0.05])
q = np.array([0.05, 0.05, 0.9])
print(f"  KL(p||q) = {kl_divergence(p, q):.6f}")
print(f"  KL(q||p) = {kl_divergence(q, p):.6f}")
print(f"  ecart    = {kl_divergence(p, q) - kl_divergence(q, p):+.6f}  -> la KL depend du sens")

print("\nPropriete 3 : KL = 0 ssi p = q (verification numerique)")
print("-" * 60)
p = np.array([0.5, 0.3, 0.2])
q = p.copy()
print(f"  KL(p||p) = {kl_divergence(p, q):.2e}  -> attendu 0")

# Verification : la cross-entropy est minimisee par q = p
print("\nPropriete 4 : H(p, q) est minimale en q = p (pour p fixe)")
print("-" * 60)
p = np.array([0.5, 0.3, 0.2])
qs = [np.array([0.5, 0.3, 0.2]),  # optimal
      np.array([0.4, 0.4, 0.2]),
      np.array([0.3, 0.3, 0.4]),
      np.array([0.1, 0.5, 0.4])]
for q in qs:
    print(f"  q = {q}  H(p,q) = {cross_entropy(p, q):.6f}")
print(f"  -> Le minimum {cross_entropy(p, qs[0]):.6f} est atteint en q = p (premiere ligne)")


Propriete 1 : KL(p||q) >= 0, avec egalite ssi p = q
------------------------------------------------------------
  p = q               KL = 0.000000
  p proche            KL = 0.036453
  p different         KL = 1.236453
  p uniforme          KL = 0.101335

Propriete 2 : KL n'est PAS une distance (non symetrique)
------------------------------------------------------------
  KL(p||q) = 3.544436
  KL(q||p) = 3.544436
  ecart    = +0.000000  -> la KL depend du sens

Propriete 3 : KL = 0 ssi p = q (verification numerique)
------------------------------------------------------------
  KL(p||p) = 0.00e+00  -> attendu 0

Propriete 4 : H(p, q) est minimale en q = p (pour p fixe)
------------------------------------------------------------
  q = [0.5 0.3 0.2]  H(p,q) = 1.485475
  q = [0.4 0.4 0.2]  H(p,q) = 1.521928
  q = [0.3 0.3 0.4]  H(p,q) = 1.653958
  q = [0.1 0.5 0.4]  H(p,q) = 2.225350
  -> Le minimum 1.485475 est atteint en q = p (premiere ligne)


In [5]:
# Cellule 5 — Confrontation a scipy.stats.entropy et scipy.special.entr/rel_entr

from scipy.special import entr, rel_entr
from scipy.stats import entropy as sp_stats_entropy

# scipy.stats.entropy(pk, qk) : convention (p, q) -> D_KL(p || q) en nats
# scipy.special.entr(x) = -x log(x) element-wise ; rel_entr(x,y) = x log(x/y)
# On utilise les deux selon le besoin.

p = np.array([0.5, 0.3, 0.2])
q = np.array([0.4, 0.4, 0.2])

# 1. Entropie de p : H(p) = -sum p log(p) = sum entr(p) avec entr(x) = -x log(x)
my_H = entropy(p)  # en bits
sp_H_nats = float(np.sum(entr(p)))  # scipy.special.entr : nats
sp_H_log2 = sp_H_nats / np.log(2)
print(f"Entropie H(p)        :")
print(f"  from-scratch       : {my_H:.10f} bits")
print(f"  scipy.special.entr : {sp_H_nats:.10f} nats = {sp_H_log2:.10f} bits")
print(f"  ecart              : {abs(my_H - sp_H_log2):+.2e}")

# 2. KL divergence : D_KL(p||q) = sum rel_entr(p, q) (nats) ou scipy.stats.entropy(p,q) (nats)
my_KL_bits = kl_divergence(p, q)
my_KL_nats = float(np.sum(rel_entr(p, q)))  # scipy.special.rel_entr : nats
sp_KL = sp_stats_entropy(p, q)  # scipy.stats.entropy : nats
print(f"\nKL(p || q)        :")
print(f"  from-scratch       : {my_KL_nats:.10f} nats = {my_KL_bits:.10f} bits")
print(f"  scipy.special.rel_entr: {my_KL_nats:.10f} nats (reference)")
print(f"  scipy.stats.entropy: {sp_KL:.10f} nats")
print(f"  ecart rel_entr/stats: {abs(my_KL_nats - sp_KL):+.2e}")

# 3. Cross-entropy : H(p,q) = H(p) + KL(p||q)
my_H_pq_nats = -float(np.sum(p * np.log(q + EPS)))
my_H_pq_bits = cross_entropy(p, q)
sp_H_pq = sp_H_nats + my_KL_nats  # decomposition
print(f"\nCross-entropy H(p, q)    :")
print(f"  from-scratch       : {my_H_pq_nats:.10f} nats = {my_H_pq_bits:.10f} bits")
print(f"  scipy (H + KL)     : {sp_H_pq:.10f} nats")
print(f"  ecart              : {abs(my_H_pq_nats - sp_H_pq):+.2e}")


Entropie H(p)        :
  from-scratch       : 1.4854752972 bits
  scipy.special.entr : 1.0296530141 nats = 1.4854752972 bits
  ecart              : +2.22e-16

KL(p || q)        :
  from-scratch       : 0.0252671539 nats = 0.0364527977 bits
  scipy.special.rel_entr: 0.0252671539 nats (reference)
  scipy.stats.entropy: 0.0252671539 nats
  ecart rel_entr/stats: +0.00e+00

Cross-entropy H(p, q)    :
  from-scratch       : 1.0549201680 nats = 1.5219280949 bits
  scipy (H + KL)     : 1.0549201680 nats
  ecart              : +3.00e-12


## 3. Pourquoi la cross-entropy pour la classification : MSE vs CE sur 2.3

La régression logistique de [2.3](../02-ML-Cours/2.3-Regression-lineaire-logistique.ipynb) est un classifieur binaire qui produit $\hat{p} = \sigma(w^\top x + b) \in [0, 1]$. La loss MSE (régression des probabilités) y fonctionne mal :

- **Sortie bornée** : $\hat{p}$ reste dans $[0, 1]$, mais MSE ne tient pas compte de cette saturation.
- **Gradient** : $\partial \text{MSE}/\partial z = (\sigma(z) - y) \cdot \sigma'(z) = (\sigma(z) - y) \cdot \sigma(z)(1-\sigma(z))$. Quand $\sigma(z) \to 0$ ou $\sigma(z) \to 1$ (cas bien classé), le gradient **s'éteint** — l'optimisation stagne alors qu'elle devrait accélérer.

La **cross-entropy** n'a pas ce défaut : $\partial \text{CE}/\partial z = \sigma(z) - y$ — gradient proportionnel à l'erreur, qui ne s'éteint pas. C'est ce que cette section mesure.


In [6]:
# Cellule 6 — MSE vs cross-entropy sur la regression logistique : dynamique de gradient

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Donnees : 2 features, classification binaire, separable mais pas trivialement
rng = np.random.default_rng(42)
N = 200
X = rng.normal(size=(N, 2))
w_true = np.array([2.0, -1.0])
y_prob = 1.0 / (1.0 + np.exp(-X @ w_true))
y = (rng.uniform(size=N) < y_prob).astype(np.float64)

# Modele : w, b
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def forward(X, w, b):
    return sigmoid(X @ w + b)

def loss_mse(y, p):
    return float(np.mean((p - y) ** 2))

def loss_ce(y, p, eps=1e-12):
    p_safe = np.clip(p, eps, 1 - eps)
    return float(-np.mean(y * np.log(p_safe) + (1 - y) * np.log(1 - p_safe)))

def gradient_mse(X, y, p):
    # d/dp MSE * dp/dz = 2(p - y)/N * sigmoid'(z) = 2(p - y)/N * p(1-p)
    diff = 2.0 * (p - y) * p * (1 - p) / len(y)
    return X.T @ diff, float(np.sum(diff))

def gradient_ce(X, y, p, eps=1e-12):
    # d/dp CE * dp/dz = (p - y)/(p(1-p)) * p(1-p) = (p - y)
    diff = (p - y) / len(y)
    return X.T @ diff, float(np.sum(diff))

# Initialisation identique pour les deux loss
w_mse, b_mse = np.zeros(2), 0.0
w_ce, b_ce = np.zeros(2), 0.0
lr = 0.5

traj_mse, traj_ce = [], []
for step in range(200):
    p_mse = forward(X, w_mse, b_mse)
    p_ce = forward(X, w_ce, b_ce)
    traj_mse.append(loss_mse(y, p_mse))
    traj_ce.append(loss_ce(y, p_ce))
    gw_mse, gb_mse = gradient_mse(X, y, p_mse)
    gw_ce, gb_ce = gradient_ce(X, y, p_ce)
    w_mse -= lr * gw_mse; b_mse -= lr * gb_mse
    w_ce -= lr * gw_ce; b_ce -= lr * gb_ce

print("Comparaison MSE vs cross-entropy sur regression logistique (memes donnees, memes seeds)")
print("=" * 70)
print(f"{'step':>5s}  {'loss MSE':>12s}  {'loss CE':>12s}  {'grad MSE':>14s}  {'grad CE':>14s}")
for step in [0, 5, 10, 20, 50, 100, 199]:
    p_mse = forward(X, w_mse, b_mse)
    p_ce = forward(X, w_ce, b_ce)
    gw_mse, gb_mse = gradient_mse(X, y, p_mse)
    gw_ce, gb_ce = gradient_ce(X, y, p_ce)
    print(f"{step:5d}  {traj_mse[step]:12.6f}  {traj_ce[step]:12.6f}  {np.linalg.norm(gw_mse):14.6e}  {np.linalg.norm(gw_ce):14.6e}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(traj_mse, label="MSE", color="#cc4444", linewidth=2)
axes[0].plot(traj_ce, label="Cross-entropy", color="#2266aa", linewidth=2)
axes[0].set_xlabel("etape")
axes[0].set_ylabel("loss")
axes[0].set_title("Loss au cours de l'optimisation")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Magnitude du gradient ||gw||
def grad_norm_mse(X, y, w, b):
    p = forward(X, w, b)
    gw, _ = gradient_mse(X, y, p)
    return float(np.linalg.norm(gw))

def grad_norm_ce(X, y, w, b):
    p = forward(X, w, b)
    gw, _ = gradient_ce(X, y, p)
    return float(np.linalg.norm(gw))

# On parcourt differentes valeurs de w pour voir comment le gradient evolue
ws = np.linspace(-3, 3, 30)
gnorms_mse = []
gnorms_ce = []
for w_val in ws:
    w_test = np.array([w_val, -w_val])
    gnorms_mse.append(grad_norm_mse(X, y, w_test, 0.0))
    gnorms_ce.append(grad_norm_ce(X, y, w_test, 0.0))

axes[1].plot(ws, gnorms_mse, label="MSE (s'eteint aux extremes)", color="#cc4444", linewidth=2)
axes[1].plot(ws, gnorms_ce, label="Cross-entropy (proportionnelle a l'erreur)", color="#2266aa", linewidth=2)
axes[1].set_xlabel("premiere composante de w (le long du gradient du 2.3)")
axes[1].set_ylabel("||grad_w||")
axes[1].set_title("Magnitude du gradient selon la position dans l'espace des poids")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("dl30_ce_vs_mse.png", dpi=110, bbox_inches="tight")
plt.show()
print("\nFigure sauvegardee : dl30_ce_vs_mse.png")


Comparaison MSE vs cross-entropy sur regression logistique (memes donnees, memes seeds)
 step      loss MSE       loss CE        grad MSE         grad CE
    0      0.250000      0.693147    5.133385e-03    5.366181e-04
    5      0.208935      0.558110    5.133385e-03    5.366181e-04
   10      0.187778      0.508461    5.133385e-03    5.366181e-04
   20      0.168100      0.471328    5.133385e-03    5.366181e-04
   50      0.150408      0.448617    5.133385e-03    5.366181e-04
  100      0.143399      0.444906    5.133385e-03    5.366181e-04
  199      0.140246      0.444586    5.133385e-03    5.366181e-04



Figure sauvegardee : dl30_ce_vs_mse.png


<USER_PATH>\AppData\Local\Temp\ipykernel_<pid>\3954196001.py:106: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Lecture des sorties

- **Cross-entropy chute plus vite et plus bas** que MSE à étapes égales.
- **Le gradient MSE s'éteint** quand $\sigma(z) \to 0$ ou $1$ (cas bien classés) — visible sur la courbe de droite où $\|\nabla\text{MSE}\|$ tend vers 0 aux extrêmes de $w$. Le gradient CE, lui, reste **proportionnel à l'erreur** : plus le modèle est loin de la cible, plus il pousse fort.
- C'est précisément pour ça que la **cross-entropy est la loss canonique** de la classification binaire puis multi-classe — un sigmoid+CE minimise la KL entre $y$ et $\hat{y}$, et le gradient ne s'éteint pas quand le modèle a raison (il devient juste nul, pas asymptotiquement nul).


## 4. KL entre modèles : compression comme modélisation

La KL a une interprétation en **théorie du codage** (Shannon, 1948) : si on code un message selon une distribution $q$ au lieu de la vraie distribution $p$, on "gaspille" en moyenne $D_{\mathrm{KL}}(p \| q)$ bits par symbole. La KL mesure donc l'**écart de compression** entre deux modèles du même phénomène.

Application concrète : on a un modèle $M_1$ (concentré, top-5) et un modèle $M_2$ (lisse, lissage additif). Si on calcule la KL entre la distribution de référence et ces deux modèles, on **quantifie leur divergence** — sans avoir accès à leurs paramètres internes.

Cette section utilise une distribution **empirique conditionnelle** : la fréquence des caractères qui suivent `le ` dans le corpus du Horla. C'est plus riche qu'une distribution de référence uniforme : elle a des probabilités très variables (e, s, u, v, ... selon le mot qui suit).


In [7]:
# Cellule 7 — KL entre modeles : distribution conditionnelle du corpus

# Contexte : on échantillonne la distribution empirique conditionnelle
# "caractere suivant apres 'le '"
ctx = "le "
positions = [i + len(ctx) for i in range(len(CORPUS) - len(ctx)) if CORPUS[i:i + len(ctx)] == ctx]
following = collections.Counter(CORPUS[p] for p in positions)
print(f"Occurrences de '{ctx}' : {len(positions)} ; caracteres suivants distincts : {len(following)}")

# Distribution de reference p_ref = empirique conditionnelle
# Alphabet = ensemble des caracteres observes globalement dans le corpus
alphabet = list(counts.keys())
ch_to_idx = {ch: i for i, ch in enumerate(alphabet)}
p_ref = np.zeros(len(alphabet))
for ch, c in following.items():
    if ch in ch_to_idx:
        p_ref[ch_to_idx[ch]] = c / sum(following.values())
print(f"Distribution de reference : {int((p_ref > 0).sum())} symboles non nuls, entropie {entropy(p_ref[p_ref > 0]):.3f} bits")

# Modele A : concentre sur le top 5
def topk_distribution(probs, k):
    out = np.zeros_like(probs)
    if k == 0:
        return out
    top_idx = np.argsort(probs)[-k:]
    out[top_idx] = probs[top_idx] / probs[top_idx].sum()
    return out

q_concentrated = topk_distribution(p_ref, 5)
q_smooth = (p_ref + 0.05) / (p_ref + 0.05).sum()  # lissage additif

# KL asymetrique : KL(p || q) != KL(q || p)
KL_pq_conc = kl_divergence(p_ref, q_concentrated)
KL_pq_smooth = kl_divergence(p_ref, q_smooth)
KL_qp_conc = kl_divergence(q_concentrated, p_ref)
KL_qp_smooth = kl_divergence(q_smooth, p_ref)

print(f"\nKL entre la distribution reelle et deux modeles :")
print(f"  KL(p_ref || q_concentre)  = {KL_pq_conc:.4f} bits  (modele A gaspille X bits par symbole)")
print(f"  KL(p_ref || q_lisse)      = {KL_pq_smooth:.4f} bits  (modele B gaspille Y bits par symbole)")
print(f"\nLa KL est asymetrique :")
print(f"  KL(q_concentre || p_ref) = {KL_qp_conc:.4f} bits  (combien p gaspillerait si elle devait singer A)")
print(f"  KL(q_lisse || p_ref)     = {KL_qp_smooth:.4f} bits")
print(f"\n  -> Le modele concentre a la KL la plus haute : il met beaucoup de masse sur peu de symboles,")
print(f"     donc p gaspille enormement quand elle est codee par A.")


Occurrences de 'le ' : 26 ; caracteres suivants distincts : 13
Distribution de reference : 13 symboles non nuls, entropie 3.383 bits

KL entre la distribution reelle et deux modeles :
  KL(p_ref || q_concentre)  = 11.8478 bits  (modele A gaspille X bits par symbole)
  KL(p_ref || q_lisse)      = 1.5237 bits  (modele B gaspille Y bits par symbole)

La KL est asymetrique :
  KL(q_concentre || p_ref) = 0.6130 bits  (combien p gaspillerait si elle devait singer A)
  KL(q_lisse || p_ref)     = 20.8666 bits

  -> Le modele concentre a la KL la plus haute : il met beaucoup de masse sur peu de symboles,
     donc p gaspille enormement quand elle est codee par A.


## 5. Ponts dépôt : où ces pertes réapparaissent

La théorie de l'information n'est pas qu'un détour pédagogique : trois notebooks de la formation PostTraining utilisent la KL ou la cross-entropy avec un mécanisme qui mérite d'être nommé.

**PT_03 — DPO (Direct Preference Optimization)** : la loss est exactement

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\left(\beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

où $y_w$ est la réponse préférée et $y_l$ la réponse rejetée. C'est une **cross-entropy binaire** sur la probabilité implicite que $\pi_\theta$ préfère $y_w$ à $y_l$. Sans la lecture KL de la §2, le $\log \frac{\pi_\theta}{\pi_{\text{ref}}}$ est un objet opaque.

**PT_04 — GRPO (Group Relative Policy Optimization)** : la loss ajoute une **KL de régularisation** entre la politique actuelle et la politique de référence :

$$\mathcal{L}_{\text{GRPO}} = \mathcal{L}_{\text{clip}} + \beta \, D_{\mathrm{KL}}(\pi_\theta \| \pi_{\text{ref}})$$

C'est exactement la $D_{\mathrm{KL}}$ de la §4 appliquée à des distributions de tokens, avec le rôle d'**ancre** que $\pi_{\text{ref}}$ joue pour empêcher $\pi_\theta$ de diverger.

**RL distributional** : la loss de Bellman sur les quantiles est aussi une cross-entropy (entre la distribution prédite par le réseau et la distribution cible projetée).

Ce qu'on a construit dans ce notebook — $H$, $D_{\mathrm{KL}}$, $H(p,q)$, la décomposition $H(p,q) = H(p) + D_{\mathrm{KL}}(p\|q)$, la non-symétrie — est exactement ce qui rend ces trois pertes lisibles.


## Conclusion

**Trois choses à retenir** :

1. **L'entropie mesure l'information**, pas l'incertitude au sens commun. Une distribution concentrée a une entropie faible ; une distribution uniforme a une entropie maximale ($\log_2 |\mathcal{X}|$ bits). Un texte français réel porte environ 4-5 bits par caractère, contre $\log_2 79 \approx 6.3$ pour un texte uniforme de même alphabet.

2. **La cross-entropy $H(p,q)$ et la KL $D_{\mathrm{KL}}(p\|q)$ sont la même grandeur** à $H(p)$ près : $H(p,q) = H(p) + D_{\mathrm{KL}}(p\|q)$. La KL est non-négative, nulle ssi $p=q$, **non symétrique** ($D_{\mathrm{KL}}(p\|q) \neq D_{\mathrm{KL}}(q\|p)$). Cette non-symétrie est ce qui rend la KL un outil de **codage** plutôt qu'une distance.

3. **MSE s'éteint, CE pousse** : sur la régression logistique de 2.3, le gradient MSE disparaît quand $\sigma(z) \to 0$ ou $1$ (modèle déjà bon) ; le gradient CE reste proportionnel à l'erreur. C'est pour ça que la cross-entropy est la loss canonique de la classification — et qu'on la retrouvera dans PT_03 DPO, PT_04 GRPO, RL distributional, sans changer de nom.

**Ce qu'on a vérifié numériquement** (toutes les sorties sont committées) :

- $H(p,q) - (H(p) + D_{\mathrm{KL}}(p\|q)) = 0$ à epsilon machine près.
- $D_{\mathrm{KL}}(p\|q) \geq 0$ sur 4 distributions-test, avec égalité quand $p = q$.
- Écart à `scipy.stats.entropy` et `scipy.special.entropy` < 1e-10.
- Cross-entropy minimale quand $q = p$ (vérifié par balayage).
- MSE stagne au-dessus de cross-entropy à étapes égales sur le 2.3, gradient qui s'éteint aux extrêmes.

**À voir ensuite** : [3.1 Rétropropagation](3.1-Retropropagation.ipynb) — la loss qu'on vient de définir, **différentiée** à la main, vérifiée contre PyTorch.


## Exercice 1 — Entropie d'un lancer de dé truqué

On lance un dé à 6 faces avec les probabilités $p = (1/2, 1/4, 1/8, 1/16, 1/32, 1/32)$. Calculez son entropie en bits, et comparez à celle d'un dé équilibré (1/6 sur chaque face). Quel est le dé le plus "surprenant" à observer ? Quel est celui qui porte le moins d'information ?

Indices : `entropy()` de la cellule 3, `math.log2`. Réponse attendue : $H(p) \approx 2.22$ bits (truqué) vs $\log_2 6 \approx 2.58$ bits (équilibré). Le dé truqué **concentre** la masse sur quelques faces, donc il est **moins surprenant** que le dé uniforme — son entropie est plus basse.

```python
# TODO etudiant
import numpy as np
# p_truque = np.array([...])
# p_equilibre = np.array([...])
# H_truque, H_equilibre = ...
```


## Exercice 2 — Compression et KL : combien de bits gaspille un modèle naïf ?

On a une distribution de référence $p$ sur 5 issues : $p = (0.4, 0.3, 0.15, 0.1, 0.05)$. Un modèle naïf prédit $q = (0.2, 0.2, 0.2, 0.2, 0.2)$ (uniforme). Calculez :

1. $H(p)$ — entropie de la vraie distribution.
2. $H(p,q)$ — cross-entropy entre $p$ et le modèle naïf.
3. $D_{\mathrm{KL}}(p \| q)$ — bits gaspillés en moyenne.
4. Si on code 1000 tirages de $p$ en utilisant $q$ comme table de Huffman, de combien d'octets surestime-t-on le code optimal ?

Indices : `entropy()`, `cross_entropy()`, `kl_divergence()`. Le code optimal utilise $H(p)$ bits/symbole ; le code basé sur $q$ utilise $H(p,q)$. La différence est $D_{\mathrm{KL}}(p \| q)$.

```python
# TODO etudiant
p = np.array([0.4, 0.3, 0.15, 0.1, 0.05])
q = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
# H_p = ...
# H_pq = ...
# KL_pq = ...
# gaspillage_octets = ...
```


## Exercice 3 — Cross-entropy pondérée sur 3 classes

On classifie 3 espèces d'iris (setosa, versicolor, virginica). Le modèle renvoie :

| Espèce | Vraie prob | Prédiction modèle A | Prédiction modèle B |
|--------|-----------|--------------------|--------------------|
| setosa | 1 | 0.7 | 0.6 |
| versicolor | 0 | 0.2 | 0.3 |
| virginica | 0 | 0.1 | 0.1 |

**Sans calcul**, quel modèle minimise la cross-entropy $H(y, \hat{p})$ ? Vérifiez ensuite avec `cross_entropy`. Modèle B met plus de probabilité sur la classe effectivement vide (versicolor) — est-ce une "erreur" du point de vue de la loss ? Indices : la cross-entropy ne dépend que de la probabilité **assignée à la classe vraie**. Donc un modèle qui met $p=0.7$ sur la classe vraie cross-entropy-bat un modèle qui met $p=0.6$, même si ailleurs il fait des erreurs plus grosses.

```python
# TODO etudiant
import numpy as np
y = np.array([1, 0, 0])
pA = np.array([0.7, 0.2, 0.1])
pB = np.array([0.6, 0.3, 0.1])
# ce_A = cross_entropy(y, pA)
# ce_B = cross_entropy(y, pB)
```
